# 06 — Final Pseudo-Absence Selection and Model Dataset Preparation

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This stage prepares the final datasets for the first machine-learning experiments.

---

# Inputs

## Static master table
From Stage 05A:

`static_feature_samples_final.csv`

## Temporal final table
From Stage 05B-QC:

`temporal_feature_samples_final_annotated.csv`

---

# Main objectives

1. Use only objectively valid static samples.
2. Keep all valid landslide positives.
3. Select final pseudo-absence/background controls at a **1:2 ratio**.
4. Preserve a **1:1 sensitivity dataset**.
5. Create 50-km spatial blocks.
6. Prepare grouped/spatial validation fields.
7. Export raw model-ready tables for:
   - Logistic Regression
   - Random Forest
   - optional geology-enhanced experiment
   - temporal secondary experiments

---

# Important modelling rules

- Pseudo-absence selection does **not** choose only low-risk environments.
- Slope, elevation, soil, fault distance, etc. are predictors, not selection filters.
- All selected controls already passed objective static-data validity checks.
- Logistic Regression and Random Forest will use the **same samples and same validation design**.
- Model training occurs in Stage 07, not here.


In [1]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)


pandas: 3.0.2


## 1. Project paths

In [2]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

STATIC_PATH = (
    DEV_DIR
    / "03_derived_features"
    / "static_features_final"
    / "static_feature_samples_final.csv"
)

TEMPORAL_PATH = (
    DEV_DIR
    / "03_derived_features"
    / "temporal_features_final"
    / "temporal_feature_samples_final_annotated.csv"
)

TEMPORAL_SUBSET_DIR = (
    DEV_DIR
    / "03_derived_features"
    / "temporal_features_final"
)

OUTPUT_DIR = (
    DEV_DIR
    / "04_model_ready"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for name, path in {
    "Static final table": STATIC_PATH,
    "Temporal annotated table": TEMPORAL_PATH,
}.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\\n{path}"
        )

    print("FOUND:", name, "->", path)

print("\\nOutput:")
print(OUTPUT_DIR)


FOUND: Static final table -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final\static_feature_samples_final.csv
FOUND: Temporal annotated table -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_feature_samples_final_annotated.csv
\nOutput:
C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready


## 2. Load final Stage-05A static table

In [3]:
static = pd.read_csv(
    STATIC_PATH
)

print("Static master rows:", len(static))

display(
    static["sample_class"]
    .value_counts()
    .rename_axis("sample_class")
    .reset_index(name="records")
)


Static master rows: 3200


,sample_class,records
0,background_candidate,3000
1,landslide,200


## 3. Select valid static positives and background candidates

The national core-static baseline requires:

- valid terrain
- valid SoilGrids
- valid major-fault distance

No imputation is performed here.


In [4]:
valid_mask = (
    static[
        "national_core_static_valid_final"
    ].eq(True)
)

positive = static.loc[
    valid_mask
    &
    static[
        "sample_class"
    ].eq(
        "landslide"
    )
].copy()

background = static.loc[
    valid_mask
    &
    static[
        "sample_class"
    ].eq(
        "background_candidate"
    )
].copy()


print(
    "Valid positive landslides:",
    len(positive)
)

print(
    "Valid background candidates:",
    len(background)
)


if len(positive) == 0:
    raise ValueError(
        "No valid positive samples."
    )

if len(background) < len(positive) * 2:
    raise ValueError(
        "Not enough valid background candidates for 1:2 sampling."
    )


Valid positive landslides: 194
Valid background candidates: 2930


# PART A — SPATIAL BLOCKS

## 4. Create approximate 50-km spatial blocks

Malaysia is close to the equator, so a simple latitude/longitude block approximation is sufficient for validation grouping here.

Approximate conversion:

- 1 degree latitude ≈ 111 km
- longitude is adjusted by latitude

For reproducibility and simplicity, a **0.45° grid** is used, approximately 50 km.

The purpose is not cartographic precision; it is to prevent immediately neighbouring samples from being freely split across training and test folds.


In [5]:
SPATIAL_BLOCK_DEG = 0.45


def spatial_block_id(
    lon,
    lat,
    block_deg=SPATIAL_BLOCK_DEG
):

    x = math.floor(
        float(lon)
        / block_deg
    )

    y = math.floor(
        float(lat)
        / block_deg
    )

    return f"B_{x}_{y}"


for dataframe in [
    positive,
    background
]:

    dataframe[
        "spatial_block_id"
    ] = dataframe.apply(
        lambda row:
            spatial_block_id(
                row[
                    "longitude"
                ],
                row[
                    "latitude"
                ]
            ),
        axis=1
    )


print(
    "Positive spatial blocks:",
    positive[
        "spatial_block_id"
    ].nunique()
)

print(
    "Background spatial blocks:",
    background[
        "spatial_block_id"
    ].nunique()
)


Positive spatial blocks: 38
Background spatial blocks: 184


# PART B — FINAL BACKGROUND / PSEUDO-ABSENCE SELECTION

## 5. Stratified proportional block sampling

The 3,000 background candidates were already generated approximately proportional to eligible Malaysian land area.

The final 1:2 selection therefore preserves that spatial distribution by allocating samples proportionally across occupied spatial blocks.

A fixed random seed makes the selection reproducible.


In [6]:
RANDOM_SEED = 42
rng = np.random.default_rng(
    RANDOM_SEED
)


def proportional_integer_allocation(
    counts,
    total_required
):
    counts = counts.astype(
        float
    )

    raw = (
        counts
        / counts.sum()
        * total_required
    )

    allocation = np.floor(
        raw
    ).astype(int)

    allocation = np.minimum(
        allocation,
        counts.astype(int)
    )

    remaining = (
        total_required
        - allocation.sum()
    )

    fractional = (
        raw
        - np.floor(
            raw
        )
    )

    # Iteratively allocate remaining samples.
    while remaining > 0:

        available = (
            counts.astype(int)
            - allocation
        )

        eligible = np.where(
            available > 0
        )[0]

        if len(eligible) == 0:
            raise RuntimeError(
                "Unable to allocate requested pseudo-absence count."
            )

        # Highest fractional remainder first.
        chosen = eligible[
            np.argmax(
                fractional[
                    eligible
                ]
            )
        ]

        allocation[
            chosen
        ] += 1

        # Avoid repeatedly choosing the same block solely by fractional score.
        fractional[
            chosen
        ] = 0

        remaining -= 1

    return allocation


def spatially_stratified_sample(
    background_df,
    n_required,
    seed
):

    local_rng = np.random.default_rng(
        seed
    )

    block_counts = (
        background_df[
            "spatial_block_id"
        ]
        .value_counts()
        .sort_index()
    )

    allocation = proportional_integer_allocation(
        block_counts.to_numpy(),
        n_required
    )

    sampled_parts = []

    allocation_rows = []

    for (
        block_id,
        available_count,
        take_n
    ) in zip(
        block_counts.index,
        block_counts.to_numpy(),
        allocation
    ):

        if take_n <= 0:
            continue

        group = background_df.loc[
            background_df[
                "spatial_block_id"
            ].eq(
                block_id
            )
        ]

        chosen = local_rng.choice(
            group.index.to_numpy(),
            size=int(
                take_n
            ),
            replace=False
        )

        sampled_parts.append(
            background_df.loc[
                chosen
            ].copy()
        )

        allocation_rows.append({
            "spatial_block_id":
                block_id,

            "available":
                int(
                    available_count
                ),

            "selected":
                int(
                    take_n
                ),
        })

    sampled = pd.concat(
        sampled_parts,
        ignore_index=False
    ).copy()

    return (
        sampled,
        pd.DataFrame(
            allocation_rows
        )
    )


## 6. Create 1:2 baseline and 1:1 sensitivity selections


In [7]:
N_POSITIVE = len(
    positive
)

N_BACKGROUND_1TO2 = (
    N_POSITIVE
    * 2
)

N_BACKGROUND_1TO1 = (
    N_POSITIVE
)


background_1to2, allocation_1to2 = (
    spatially_stratified_sample(
        background,
        N_BACKGROUND_1TO2,
        seed=RANDOM_SEED
    )
)


# 1:1 is selected as a subset of 1:2.
background_1to1, allocation_1to1 = (
    spatially_stratified_sample(
        background_1to2,
        N_BACKGROUND_1TO1,
        seed=RANDOM_SEED + 1
    )
)


print(
    "Positive:",
    N_POSITIVE
)

print(
    "1:2 background:",
    len(
        background_1to2
    )
)

print(
    "1:1 background:",
    len(
        background_1to1
    )
)


Positive: 194
1:2 background: 388
1:1 background: 194


## 7. Build final static classification datasets

For modelling:

- landslide = `target 1`
- selected background/pseudo-absence = `target 0`

The term `pseudo_absence` is used because these are not verified never-landslide locations.


In [8]:
positive_model = positive.copy()

positive_model[
    "target"
] = 1

positive_model[
    "model_class"
] = "landslide"


def prepare_background(
    background_df
):

    out = background_df.copy()

    out[
        "target"
    ] = 0

    out[
        "model_class"
    ] = "pseudo_absence"

    return out


background_1to2_model = prepare_background(
    background_1to2
)

background_1to1_model = prepare_background(
    background_1to1
)


static_1to2 = pd.concat(
    [
        positive_model,
        background_1to2_model
    ],
    ignore_index=True,
    sort=False
)


static_1to1 = pd.concat(
    [
        positive_model,
        background_1to1_model
    ],
    ignore_index=True,
    sort=False
)


print(
    "1:2 total rows:",
    len(
        static_1to2
    )
)

print(
    "1:1 total rows:",
    len(
        static_1to1
    )
)


1:2 total rows: 582
1:1 total rows: 388


# PART C — PREDICTOR DEFINITIONS

## 8. Define first-experiment predictor sets

### Core static predictors

- elevation
- slope
- aspect encoded as sine/cosine
- SoilGrids clay
- SoilGrids sand
- SoilGrids silt
- SoilGrids bulk density
- distance to major fault

Aspect is circular, so raw degrees are converted to:

- `aspect_sin`
- `aspect_cos`

This avoids treating 1° and 359° as far apart.


In [9]:
for dataframe in [
    static_1to2,
    static_1to1
]:

    aspect_rad = np.deg2rad(
        pd.to_numeric(
            dataframe[
                "aspect_deg"
            ],
            errors="coerce"
        )
    )

    dataframe[
        "aspect_sin"
    ] = np.sin(
        aspect_rad
    )

    dataframe[
        "aspect_cos"
    ] = np.cos(
        aspect_rad
    )


CORE_STATIC_FEATURES = [
    "elevation_m",
    "slope_deg",
    "aspect_sin",
    "aspect_cos",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
]


print(
    "Core static predictors:"
)

for feature in CORE_STATIC_FEATURES:
    print(" -", feature)


Core static predictors:
 - elevation_m
 - slope_deg
 - aspect_sin
 - aspect_cos
 - soil_clay
 - soil_sand
 - soil_silt
 - soil_bulk_density
 - distance_major_fault_m


## 9. Optional geology-enhanced fields

These are preserved in the raw dataset.

Categorical encoding should be performed inside the model pipeline during Stage 07 to avoid leakage.

Candidate geology fields:

- `geology_age`
- `geology_unit_type`
- potentially `geology_unit_name` if category counts are manageable

Do not one-hot encode before cross-validation.


In [10]:
GEOLOGY_CATEGORICAL_FEATURES = [
    col
    for col in [
        "geology_age",
        "geology_unit_type",
    ]
    if col in static_1to2.columns
]


print(
    "Available geology categorical predictors:"
)

print(
    GEOLOGY_CATEGORICAL_FEATURES
)


Available geology categorical predictors:
['geology_age', 'geology_unit_type']


# PART D — SPATIAL VALIDATION PREPARATION

## 10. Validate spatial-block composition

Stage 07 will use the same spatial groups for both Logistic Regression and Random Forest.

Preferred first design:

`GroupKFold(n_splits=5)`

with:

`groups = spatial_block_id`

No random row-level train/test split will be used as the primary evaluation.


In [11]:
block_class_summary = (
    static_1to2.groupby(
        [
            "spatial_block_id",
            "target"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)


# Standard column names.
if 0 not in block_class_summary.columns:
    block_class_summary[0] = 0

if 1 not in block_class_summary.columns:
    block_class_summary[1] = 0


block_class_summary = (
    block_class_summary.rename(
        columns={
            0:
                "pseudo_absence_count",

            1:
                "landslide_count",
        }
    )
)


block_class_summary[
    "total"
] = (
    block_class_summary[
        "pseudo_absence_count"
    ]
    +
    block_class_summary[
        "landslide_count"
    ]
)


display(
    block_class_summary.sort_values(
        "total",
        ascending=False
    ).head(30)
)


print(
    "Total spatial blocks:",
    len(
        block_class_summary
    )
)


print(
    "Blocks containing positives:",
    int(
        block_class_summary[
            "landslide_count"
        ]
        .gt(0)
        .sum()
    )
)


target,spatial_block_id,pseudo_absence_count,landslide_count,total
29,B_226_7,3,34,37
22,B_225_9,3,23,26
28,B_226_6,3,22,25
19,B_225_6,2,15,17
0,B_222_11,0,16,16
20,B_225_7,3,13,16
1,B_222_12,0,14,14
140,B_259_13,3,8,11
134,B_258_13,2,7,9
16,B_225_10,3,4,7


Total spatial blocks: 167
Blocks containing positives: 38


## 11. Spatial-validation feasibility checks


In [12]:
N_SPATIAL_FOLDS = 5


if (
    static_1to2[
        "spatial_block_id"
    ]
    .nunique()
    < N_SPATIAL_FOLDS
):

    raise ValueError(
        "Not enough spatial blocks for 5-fold grouped validation."
    )


positive_blocks = (
    static_1to2.loc[
        static_1to2[
            "target"
        ].eq(
            1
        ),
        "spatial_block_id"
    ]
    .nunique()
)


if positive_blocks < N_SPATIAL_FOLDS:

    raise ValueError(
        "Too few positive-containing blocks for 5-fold validation."
    )


print(
    "Spatial GroupKFold feasibility: PASS"
)

print(
    "Planned folds:",
    N_SPATIAL_FOLDS
)


Spatial GroupKFold feasibility: PASS
Planned folds: 5


# PART E — SAVE STATIC MODEL DATASETS

## 12. Save final static datasets


In [13]:
static_1to2_path = (
    OUTPUT_DIR
    / "static_model_1to2.csv"
)

static_1to1_path = (
    OUTPUT_DIR
    / "static_model_1to1_sensitivity.csv"
)


static_1to2.to_csv(
    static_1to2_path,
    index=False
)

static_1to1.to_csv(
    static_1to1_path,
    index=False
)


allocation_1to2.to_csv(
    OUTPUT_DIR
    / "pseudoabsence_1to2_spatial_allocation.csv",
    index=False
)


allocation_1to1.to_csv(
    OUTPUT_DIR
    / "pseudoabsence_1to1_spatial_allocation.csv",
    index=False
)


block_class_summary.to_csv(
    OUTPUT_DIR
    / "static_spatial_block_class_summary.csv",
    index=False
)


print(
    "Saved:",
    static_1to2_path
)

print(
    "Saved:",
    static_1to1_path
)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to2.csv
Saved: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to1_sensitivity.csv


# PART F — TEMPORAL MODEL DATASETS

## 13. Load Stage-05B final temporal subsets

These already preserve the 1:2 matched case-control design.


In [14]:
temporal_files = {
    "rainfall":
        TEMPORAL_SUBSET_DIR
        / "temporal_model_rainfall.csv",

    "rainfall_soilmoisture":
        TEMPORAL_SUBSET_DIR
        / "temporal_model_rainfall_soilmoisture.csv",

    "landcover":
        TEMPORAL_SUBSET_DIR
        / "temporal_model_landcover.csv",

    "full_enhanced":
        TEMPORAL_SUBSET_DIR
        / "temporal_model_full_enhanced.csv",
}


temporal_datasets = {}


for name, path in temporal_files.items():

    if path.exists():

        temp = pd.read_csv(
            path
        )

        temporal_datasets[
            name
        ] = temp

        print(
            name,
            ":",
            len(
                temp
            ),
            "rows"
        )

    else:

        print(
            "MISSING:",
            name,
            "->",
            path
        )


rainfall : 567 rows
rainfall_soilmoisture : 474 rows
landcover : 444 rows
full_enhanced : 363 rows


## 14. Add spatial block IDs to temporal datasets

For matched temporal experiments, each matched group must remain intact.

Therefore the validation group is based on the **case landslide's spatial block**.

Every control matched to that case inherits the case's validation block.


In [15]:
def assign_temporal_case_blocks(
    dataframe
):

    out = dataframe.copy()

    case_blocks = {}

    cases = out.loc[
        out[
            "temporal_role"
        ].eq(
            "case"
        )
    ]

    for _, row in cases.iterrows():

        case_blocks[
            row[
                "matched_positive_id"
            ]
        ] = spatial_block_id(
            row[
                "longitude"
            ],
            row[
                "latitude"
            ]
        )


    out[
        "case_spatial_block_id"
    ] = out[
        "matched_positive_id"
    ].map(
        case_blocks
    )


    out[
        "validation_group"
    ] = out[
        "case_spatial_block_id"
    ]


    return out


for name in list(
    temporal_datasets.keys()
):

    temporal_datasets[
        name
    ] = assign_temporal_case_blocks(
        temporal_datasets[
            name
        ]
    )

    print(
        name,
        "| validation blocks:",
        temporal_datasets[
            name
        ][
            "validation_group"
        ].nunique()
    )


rainfall | validation blocks: 38
rainfall_soilmoisture | validation blocks: 34
landcover | validation blocks: 33
full_enhanced | validation blocks: 28


## 15. Save temporal model-ready raw tables

These are still unencoded.

All preprocessing must occur **inside each training fold** in Stage 07.


In [16]:
for name, dataframe in temporal_datasets.items():

    path = (
        OUTPUT_DIR
        / f"temporal_model_{name}_spatial.csv"
    )

    dataframe.to_csv(
        path,
        index=False
    )

    print(
        "Saved:",
        path
    )


Saved: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_rainfall_spatial.csv
Saved: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_rainfall_soilmoisture_spatial.csv
Saved: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_landcover_spatial.csv
Saved: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_full_enhanced_spatial.csv


# PART G — FINAL MODEL-READINESS SUMMARY

## 16. Dataset summary


In [17]:
summary_rows = [
    {
        "dataset":
            "Static baseline 1:2",

        "rows":
            len(
                static_1to2
            ),

        "positives":
            int(
                static_1to2[
                    "target"
                ]
                .eq(
                    1
                )
                .sum()
            ),

        "controls":
            int(
                static_1to2[
                    "target"
                ]
                .eq(
                    0
                )
                .sum()
            ),

        "validation_groups":
            static_1to2[
                "spatial_block_id"
            ]
            .nunique(),
    },

    {
        "dataset":
            "Static sensitivity 1:1",

        "rows":
            len(
                static_1to1
            ),

        "positives":
            int(
                static_1to1[
                    "target"
                ]
                .eq(
                    1
                )
                .sum()
            ),

        "controls":
            int(
                static_1to1[
                    "target"
                ]
                .eq(
                    0
                )
                .sum()
            ),

        "validation_groups":
            static_1to1[
                "spatial_block_id"
            ]
            .nunique(),
    },
]


for name, dataframe in temporal_datasets.items():

    summary_rows.append({
        "dataset":
            f"Temporal {name}",

        "rows":
            len(
                dataframe
            ),

        "positives":
            int(
                dataframe[
                    "temporal_role"
                ]
                .eq(
                    "case"
                )
                .sum()
            ),

        "controls":
            int(
                dataframe[
                    "temporal_role"
                ]
                .eq(
                    "matched_control"
                )
                .sum()
            ),

        "validation_groups":
            dataframe[
                "validation_group"
            ]
            .nunique(),
    })


model_dataset_summary = pd.DataFrame(
    summary_rows
)


display(
    model_dataset_summary
)


model_dataset_summary.to_csv(
    OUTPUT_DIR
    / "model_dataset_summary.csv",
    index=False
)


,dataset,rows,positives,controls,validation_groups
0,Static baseline 1:2,582,194,388,167
1,Static sensitivity 1:1,388,194,194,149
2,Temporal rainfall,567,189,378,38
3,Temporal rainfall_soilmoisture,474,158,316,34
4,Temporal landcover,444,148,296,33
5,Temporal full_enhanced,363,121,242,28


## 17. Save model configuration

This file documents the decisions that Stage 07 must preserve.


In [18]:
configuration = pd.DataFrame([
    {
        "setting":
            "Primary static ratio",

        "value":
            "1:2"
    },

    {
        "setting":
            "Sensitivity static ratio",

        "value":
            "1:1"
    },

    {
        "setting":
            "Primary validation",

        "value":
            "5-fold spatial GroupKFold"
    },

    {
        "setting":
            "Spatial block size",

        "value":
            "approximately 50 km (0.45 degrees)"
    },

    {
        "setting":
            "Baseline models",

        "value":
            "Logistic Regression; Random Forest"
    },

    {
        "setting":
            "Core static numerical predictors",

        "value":
            "; ".join(
                CORE_STATIC_FEATURES
            )
    },

    {
        "setting":
            "Geology experiment",

        "value":
            "Optional; encode inside CV pipeline"
    },

    {
        "setting":
            "Historical road",

        "value":
            "Optional; not required for first baseline"
    },
])


display(
    configuration
)


configuration.to_csv(
    OUTPUT_DIR
    / "model_configuration.csv",
    index=False
)


,setting,value
0,Primary static ratio,1:2
1,Sensitivity static ratio,1:1
2,Primary validation,5-fold spatial GroupKFold
3,Spatial block size,approximately 50 km (0.45 degrees)
4,Baseline models,Logistic Regression; Random Forest
5,Core static numerical predictors,elevation_m; slope_deg; aspect_sin; aspect_cos...
6,Geology experiment,Optional; encode inside CV pipeline
7,Historical road,Optional; not required for first baseline


# Stage 06 completion

The next notebook should be:

## `07_baseline_model_training.ipynb`

It will train:

1. **Logistic Regression**
2. **Random Forest**

using exactly the same:

- static 1:2 sample dataset
- 5-fold spatial GroupKFold splits
- predictor set

Metrics:

- accuracy
- balanced accuracy
- precision
- recall
- F1-score
- ROC-AUC
- confusion matrix

The 1:1 static dataset will be retained for sensitivity analysis.

XGBoost should only be considered **after** the Logistic Regression and Random Forest baselines are working.


# Outputs to send for review

Please send:

```text
model_dataset_summary.csv
model_configuration.csv
static_spatial_block_class_summary.csv
pseudoabsence_1to2_spatial_allocation.csv
```

You do not need to upload every model-ready CSV yet unless an error appears.
